# Dataset audit for the motor-condition project (read-only)
Run the cells **in order, top to bottom**. This notebook **never edits your original file**: it reads it, prints a report and saves the report in a separate folder.

**What you need:** `motor_predictive_maintenance_master.csv` (the original two-header-row file). Optional: `motor_clean_working_copy.csv` and the model bundle folder from your training notebook.

**What the verdict means:** the tool will **not** declare the dataset "valid". The best it can say is *structure checks passed, label provenance UNVERIFIED*, because only the data owner can say who assigned Normal/Alert/Danger.

## Step 1. Install the audit tools (about 10 seconds)
The next cells only write files into Colab. Run each one; nothing is analysed yet.

In [ ]:
import os
os.makedirs('tests', exist_ok=True)
!pip -q install pytest scikit-learn pandas numpy

In [ ]:
%%writefile pm_data_audit.py
#!/usr/bin/env python3
"""pm_data_audit.py - read-only data-quality and leakage audit for the motor condition dataset.

WHAT IT DOES
  Runs independent checks on the RAW master CSV (two header rows, 18 columns) and, optionally, on the
  cleaned working copy and the exported model bundle. It has its OWN loader on purpose: if it reused the
  training notebook's loader, a bug there would be invisible here.

WHAT IT NEVER DOES
  * It never writes to, renames or edits the input file (SHA-256 is recorded before and after).
  * It never fills, drops or invents values or labels.
  * It never says "valid". The strongest verdict is "structure checks passed, label provenance UNVERIFIED",
    because no program can verify who assigned the labels.

USAGE (Colab or any machine)
  python pm_data_audit.py --data motor_predictive_maintenance_master.csv --out audit_outputs
  python pm_data_audit.py --data RAW.csv --clean-copy motor_clean_working_copy.csv --bundle pm_condition_ann_v1.0.0 --out audit_outputs

Exit code: 0 = no FAIL, 1 = at least one FAIL, 2 = could not run.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import re
import sys
from collections import OrderedDict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

AUDIT_VERSION = "1.0.0"

# ----------------------------------------------------------------------------------------------
# Contract copied from the inspection report of the real file (NOT from the notebook loader).
# ----------------------------------------------------------------------------------------------
ID_HEADERS = ["Record ID", "Motor ID", "Measurement timestamps"]
# raw sub-heading -> (clean name used by the notebook/backend, unit)
SENSORS = OrderedDict([
    ("DE_H_mm/s", ("de_h_velocity_mm_s", "mm/s")), ("DE_H_gE", ("de_h_ge", "gE")), ("DE_H_temp_C", ("de_h_temp_c", "degC")),
    ("DE_V_mm/s", ("de_v_velocity_mm_s", "mm/s")), ("DE_V_gE", ("de_v_ge", "gE")), ("DE_V_temp_C", ("de_v_temp_c", "degC")),
    ("NDE_H_mm/s", ("nde_h_velocity_mm_s", "mm/s")), ("NDE_H_gE", ("nde_h_ge", "gE")), ("NDE_H_temp_C", ("nde_h_temp_c", "degC")),
    ("NDE_V_mm/s", ("nde_v_velocity_mm_s", "mm/s")), ("NDE_V_gE", ("nde_v_ge", "gE")), ("NDE_V_temp_C", ("nde_v_temp_c", "degC")),
    ("rotational_speed_rpm", ("rotational_speed_rpm", "rpm")), ("load_percent", ("load_percent", "%")),
])
LABEL_RAW = "condition_label"
CLASSES = ["Normal", "Alert", "Danger"]
SUBHEADINGS = list(SENSORS) + [LABEL_RAW]
CLEAN = [v[0] for v in SENSORS.values()]
FEATURES = CLEAN                                  # the order the backend must use
TS_MDY, TS_DMY = "%m/%d/%Y %H:%M", "%d/%m/%Y %H:%M"

# What the earlier inspection of the real file reported (used only to detect "this is a different file").
INSPECTED = {"sha256": "0270fc74bc9f428b97a0ab8c9cad22fa92a5916edeea98ea9fdf93a795891a8b", "bytes": 216691,
             "rows": 1938, "motors": 40, "labels": {"Normal": 1475, "Alert": 347, "Danger": 116}}

# Generic engineering screens. They only raise WARN for a human to review. They are NOT validity rules,
# and nothing is ever removed because of them.
PLAUSIBLE = {"mm/s": (0.0, 50.0), "gE": (0.0, 100.0), "degC": (-20.0, 200.0), "rpm": (1.0, 4000.0), "%": (0.0, 150.0)}
SENTINELS = {-9999.0, -999.0, -99.0, -9.0, 999.0, 9999.0, 99999.0}
FAIL, WARN, PASS, INFO, SKIP = "FAIL", "WARN", "PASS", "INFO", "SKIP"

# Reproduces notebook v1.0.0 section 11 (split by motors, label counts only). Re-implemented here so the
# audit can check the split WITHOUT importing the notebook. Compare its table with the one the notebook printed.
SEED, SPLIT_FRACTIONS, MIN_CLASS_RECORDS, N_SPLIT_TRIALS = 42, (0.60, 0.20, 0.20), 10, 5000


class Audit:
    def __init__(self):
        self.findings: list[dict] = []
        self.tables: dict[str, object] = {}

    def add(self, fid, title, status, detail, evidence=None):
        self.findings.append({"id": fid, "title": title, "status": status, "detail": detail, "evidence": evidence or {}})
        return status

    def worst(self):
        order = [FAIL, WARN, INFO, SKIP, PASS]
        return min((f["status"] for f in self.findings), key=order.index, default=PASS)


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ----------------------------------------------------------------------------------------------
# 1. File and structure
# ----------------------------------------------------------------------------------------------
def read_text_grid(path, a: Audit):
    raw = Path(path).read_bytes()
    bom = raw.startswith(b"\xef\xbb\xbf")
    nul, crlf = raw.count(b"\x00"), raw.count(b"\r\n")
    lf_only = raw.count(b"\n") - crlf
    sha0 = sha256_of(path)
    a.add("F01", "File fingerprint", INFO, f"{len(raw):,} bytes; SHA-256 {sha0}; UTF-8 BOM: {bom}; NUL bytes: {nul}; CRLF lines: {crlf}; LF-only lines: {lf_only}",
          {"sha256": sha0, "bytes": len(raw)})
    try:
        text = raw.decode("utf-8-sig")
    except UnicodeDecodeError as exc:
        a.add("F02", "Text encoding", FAIL, f"File is not valid UTF-8: {exc}")
        text = raw.decode("latin-1")
    else:
        a.add("F02", "Text encoding", PASS, "Decodes cleanly as UTF-8.")
    sha = sha256_of(path)
    if sha == INSPECTED["sha256"]:
        a.add("F03", "Same file as the earlier inspection", PASS, "SHA-256 equals the file described in the inspection report.")
    else:
        a.add("F03", "Same file as the earlier inspection", WARN,
              "SHA-256 differs from the inspected file. This is a DIFFERENT or EDITED file; every earlier number may be stale. "
              f"inspected={INSPECTED['sha256'][:16]}..., now={sha[:16]}...")
    return list(csv.reader(io.StringIO(text))), sha


def split_header_and_body(rows, a: Audit):
    lens = pd.Series([len(r) for r in rows]).value_counts().to_dict()
    blank = sum(1 for r in rows if not any(c.strip() for c in r))
    ragged = [(i + 1, len(r)) for i, r in enumerate(rows) if any(c.strip() for c in r) and len(r) != 18]
    a.add("S01", "Column count per row", PASS if not ragged else FAIL,
          f"Field counts per row: {lens}. Blank lines: {blank}. Rows with a field count other than 18: {len(ragged)} (first: {ragged[:5]}).")
    rows = [r for r in rows if any(c.strip() for c in r)]
    if len(rows) < 3:
        a.add("S02", "Header rows", FAIL, "Fewer than 3 non-blank rows.")
        return None
    r0, r1 = [c.strip() for c in rows[0]], [c.strip() for c in rows[1]]
    ok0 = r0[:3] == ID_HEADERS
    ok1 = r1[3:18] == SUBHEADINGS and not any(r1[:3])
    a.add("S02", "Two-row header matches the contract", PASS if ok0 and ok1 else FAIL,
          f"row0 id headers {'OK' if ok0 else 'DIFFER: ' + str(r0[:3])}; row1 sub-headings {'OK' if ok1 else 'DIFFER'}"
          + ("" if ok1 else f" (expected {SUBHEADINGS}, found {r1[3:]})"))
    a.add("S03", "Group headers sit over the right blocks", PASS if all(r0[c].lower().startswith(p) for c, p in
          {3: "drive end", 6: "drive end", 9: "non drive end", 12: "non drive end"}.items() if c < len(r0)) else FAIL,
          f"row0 columns 3,6,9,12 = {[r0[c] for c in (3, 6, 9, 12) if c < len(r0)]}. 'Header'/'Vrtex' as typed are ASSUMED to mean horizontal/vertical (unconfirmed).")
    body = pd.DataFrame([r + [""] * (18 - len(r)) if len(r) < 18 else r[:18] for r in rows[2:]],
                        columns=["record_id", "motor_id", "ts_raw"] + SUBHEADINGS).apply(lambda s: s.str.strip())
    body.insert(0, "file_row", range(3, 3 + len(body)))      # 1-based line number among non-blank rows
    a.add("S04", "Data rows", PASS if len(body) == INSPECTED["rows"] else WARN,
          f"{len(body)} data rows (earlier inspection: {INSPECTED['rows']}).")
    return body


# ----------------------------------------------------------------------------------------------
# 2. Identifiers
# ----------------------------------------------------------------------------------------------
def check_ids(b, a: Audit):
    rid, mid = b["record_id"], b["motor_id"]
    empty_r, empty_m = int((rid == "").sum()), int((mid == "").sum())
    dup = rid[rid.duplicated(keep=False) & (rid != "")]
    a.add("I01", "Record IDs present and unique", PASS if not (empty_r or len(dup)) else FAIL,
          f"empty: {empty_r}; duplicated values: {dup.nunique()} covering {len(dup)} rows; examples: {sorted(dup.unique())[:5]}")
    bad_pat = rid[(rid != "") & ~rid.str.fullmatch(r"REC\d{6}")]
    a.add("I02", "Record ID pattern", PASS if bad_pat.empty else WARN, f"{len(bad_pat)} IDs do not match REC######; examples {bad_pat.head(5).tolist()}")
    nums = pd.to_numeric(rid.str.extract(r"(\d+)$")[0], errors="coerce")
    gaps = int((nums.diff().dropna() != 1).sum())
    a.add("I03", "Record ID sequence follows file order", PASS if gaps == 0 else INFO,
          f"{gaps} places where the ID does not increase by 1 in file order. Because the file is ordered motor-then-time, a record number "
          f"encodes motor and time: it must never be a model input.")
    norm = mid.str.upper().str.replace(r"[\s_\-]", "", regex=True)
    variants = {k: sorted(set(mid[norm == k])) for k in norm.unique() if len(set(mid[norm == k])) > 1}
    ws = sorted(set(mid[mid != mid.str.strip()]))
    a.add("M01", "Motor IDs present and consistently spelled", PASS if not (empty_m or variants) else FAIL,
          f"empty: {empty_m}; IDs that collapse together after removing case/space/dash: {variants}")
    per = mid[mid != ""].value_counts()
    fam = mid.str.replace(r"\d+$", "", regex=True)
    a.add("M02", "Motors and records per motor", PASS if per.min() >= 20 else WARN,
          f"{mid.nunique()} motors (earlier inspection: {INSPECTED['motors']}); records per motor min/median/max = {per.min()}/{int(per.median())}/{per.max()}; "
          f"motor families by ID prefix: {fam.groupby(fam).apply(lambda s: mid[s.index].nunique()).to_dict()}")
    return fam


# ----------------------------------------------------------------------------------------------
# 3. Timestamps and chronology
# ----------------------------------------------------------------------------------------------
def _parse(v, fmt):
    try:
        return datetime.strptime(v, fmt)
    except ValueError:
        return None


def check_timestamps(b, a: Audit):
    raw = b["ts_raw"]
    mdy, dmy = raw.map(lambda v: _parse(v, TS_MDY)), raw.map(lambda v: _parse(v, TS_DMY))
    empty = int((raw == "").sum())
    neither = b[(raw != "") & mdy.isna() & dmy.isna()]
    only_m, only_d = int((mdy.notna() & dmy.isna()).sum()), int((dmy.notna() & mdy.isna()).sum())
    both = int((mdy.notna() & dmy.notna()).sum())
    a.add("T01", "Every timestamp parses", PASS if not (empty or len(neither)) else FAIL,
          f"empty: {empty}; unparseable by BOTH month-first and day-first: {len(neither)} (examples {neither.ts_raw.head(5).tolist()}; file rows {neither.file_row.head(5).tolist()})")
    # decisive evidence for month-first vs day-first: chronological order within each motor in FILE order
    def decreases(series):
        n = 0
        for m, idx in b.groupby("motor_id").groups.items():
            t = series.loc[idx].dropna()
            n += int((t.diff().dropna() < pd.Timedelta(0)).sum()) if len(t) > 1 else 0
        return n
    tm, td = pd.to_datetime(mdy, errors="coerce"), pd.to_datetime(dmy, errors="coerce")
    dec_m, dec_d = decreases(tm), decreases(td)
    verdict = ("MONTH-first supported (fewer order violations)" if dec_m < dec_d else "DAY-first supported" if dec_d < dec_m else "not decided by file order")
    a.add("T02", "Day/month order (month-first assumed by the notebook)", PASS if only_d == 0 and dec_m <= dec_d else WARN,
          f"rows readable only as day>12 in 2nd field (month-first only): {only_m}; only as day-first: {only_d}; ambiguous (both valid): {both}. "
          f"File-order decreases within a motor: month-first={dec_m}, day-first={dec_d} -> {verdict}. Still confirm with the data owner.",
          {"decreases_month_first": dec_m, "decreases_day_first": dec_d})
    a.add("T03", "One timestamp convention in the whole file", FAIL if only_d else PASS,
          f"{only_d} rows can only be day-first and {only_m} can only be month-first." + (" The file mixes conventions." if only_d else " No row contradicts month-first."))
    t = tm.copy(); b = b.assign(t=t)
    if t.notna().any():
        now = pd.Timestamp.now()
        a.add("T04", "Timestamp range is plausible", PASS if (t.max() <= now and t.min().year >= 2000) else WARN,
              f"{t.min()} to {t.max()} ({(t.max() - t.min()).days} days). In the future: {int((t > now).sum())}. "
              f"No timezone and minute resolution only (INFO): ties and daylight-saving shifts cannot be excluded.")
    dsame = b[b.duplicated(["motor_id", "t"], keep=False) & b.t.notna()]
    a.add("T05", "No two readings of one motor share a timestamp", PASS if dsame.empty else FAIL,
          f"{len(dsame)} rows share motor+timestamp: file rows {dsame.file_row.head(10).tolist()}")
    cross = int(b.duplicated("t", keep=False).sum()) - len(dsame)
    a.add("T06", "Timestamp collisions across different motors", INFO, f"{cross} rows share a timestamp with another motor (normal if rounds are logged together).")
    out_of_order, ties, gaps_all = [], 0, []
    for m, g in b.groupby("motor_id", sort=False):
        d = g["t"].diff().dropna()
        if (d < pd.Timedelta(0)).any():
            out_of_order.append((m, int((d < pd.Timedelta(0)).sum())))
        ties += int((d == pd.Timedelta(0)).sum()); gaps_all += list(d[d > pd.Timedelta(0)].dt.total_seconds() / 86400)
    a.add("T07", "Chronological order within each motor (file order)", PASS if not out_of_order else FAIL,
          f"motors with a reading earlier than the previous one: {out_of_order[:8]}. Notebook sorts by motor+time+record_id, so sorting hides this from its own checks.")
    g = np.array(gaps_all) if gaps_all else np.array([np.nan])
    a.add("T08", "Sampling regularity", INFO if np.nanmax(g) <= 21 else WARN,
          f"gap between consecutive readings of a motor: median {np.nanmedian(g):.1f} d, 95th pct {np.nanpercentile(g, 95):.1f} d, max {np.nanmax(g):.1f} d; "
          f"gaps > 14 d: {int((g > 14).sum())}. Readings are periodic rounds, not continuous logging.")
    return b


# ----------------------------------------------------------------------------------------------
# 4. Numbers, units, duplicates
# ----------------------------------------------------------------------------------------------
NUM_RE = re.compile(r"^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$")


def check_numbers(b, a: Audit):
    X = pd.DataFrame(index=b.index)
    problems, empties, sent, implaus, neg = {}, {}, {}, {}, {}
    for raw_name, (clean, unit) in SENSORS.items():
        s = b[raw_name]
        e = int((s == "").sum()); empties[raw_name] = e
        bad = s[(s != "") & ~s.map(lambda v: bool(NUM_RE.match(v)))]
        if len(bad):
            problems[raw_name] = bad.value_counts().head(5).to_dict()
        v = pd.to_numeric(s.where(s.map(lambda x: bool(NUM_RE.match(x)))), errors="coerce")
        X[clean] = v
        lo, hi = PLAUSIBLE[unit]
        sent[raw_name] = int(v.isin(SENTINELS).sum())
        implaus[raw_name] = int(((v < lo) | (v > hi)).sum()) + int((v == 0).sum() if unit in ("mm/s", "gE", "rpm") else 0)
        neg[raw_name] = int((v < 0).sum())
    a.add("N01", "No missing measurement cells", PASS if not any(empties.values()) else FAIL, f"empty cells per column: { {k: v for k, v in empties.items() if v} or 'none' }")
    a.add("N02", "Every non-empty cell is a plain number", PASS if not problems else FAIL, f"non-numeric text: {problems or 'none'} (NaN/inf/'N/A'/'-' strings are caught here)")
    a.add("N03", "No sentinel codes (e.g. -999, 9999) hiding as numbers", PASS if not any(sent.values()) else WARN, f"counts: { {k: v for k, v in sent.items() if v} or 'none' }")
    a.add("N04", "No negative physical readings", PASS if not any(neg.values()) else FAIL, f"negative counts: { {k: v for k, v in neg.items() if v} or 'none' }")
    a.add("N05", "Generic plausibility screen (review only, nothing removed)", PASS if not any(implaus.values()) else WARN,
          f"values outside {PLAUSIBLE} or exactly zero (mm/s, gE, rpm): { {k: v for k, v in implaus.items() if v} or 'none' }")
    desc = X.describe().T[["min", "50%", "max"]].round(3)
    a.tables["numeric_ranges"] = desc.reset_index().rename(columns={"index": "feature"}).to_dict("records")
    # constant / stuck sensors
    const_cols = [c for c in CLEAN if X[c].nunique(dropna=True) <= 1]
    stuck = 0
    for m, g in b.groupby("motor_id"):
        v = X.loc[g.index, CLEAN]
        stuck += int((v.diff().iloc[1:].abs().sum(axis=1) == 0).sum())
    a.add("N06", "No constant column or stuck repeated readings", PASS if not const_cols and stuck == 0 else WARN,
          f"constant columns: {const_cols or 'none'}; consecutive identical full 14-value readings of the same motor: {stuck}")
    return X


def check_units(b, X, fam, a: Audit):
    med = X.groupby(b["motor_id"]).median()
    sensor_cols = [c for c in CLEAN if c not in ("rotational_speed_rpm", "load_percent")]   # operating conditions are covered by U02
    z = (med[sensor_cols] - med[sensor_cols].median()) / (1.4826 * (med[sensor_cols] - med[sensor_cols].median()).abs().median().replace(0, np.nan))
    flagged = {m: {c: round(float(z.loc[m, c]), 1) for c in sensor_cols if abs(z.loc[m, c]) > 6} for m in med.index}
    flagged = {m: v for m, v in flagged.items() if v}
    ratio = (med[[c for c in CLEAN if "velocity" in c]] / med[[c for c in CLEAN if "velocity" in c]].median())
    off = ratio[(ratio < 0.2) | (ratio > 5)].stack().to_dict()
    a.add("U01", "Motors whose typical reading is far from the other motors (possible unit/scale problem)", PASS if not flagged and not off else WARN,
          f"robust z>6 (motor median vs median of motors): {flagged or 'none'}; velocity medians <0.2x or >5x the fleet: {off or 'none'}. "
          f"A WARN is a question for the data owner (different unit? different sensor? genuinely different motor?), not proof of error.")
    fam_tbl = X.groupby(fam).median().round(3)
    a.tables["family_medians"] = fam_tbl.reset_index().rename(columns={"motor_id": "family", 0: "family"}).to_dict("records")
    rpm = b.assign(rpm=X["rotational_speed_rpm"]).groupby("motor_id")["rpm"].median()
    a.add("U02", "Rotational speed groups", INFO,
          f"per-motor median rpm min/max {rpm.min():.0f}/{rpm.max():.0f}; histogram by 250-rpm bins {np.histogram(rpm, bins=range(0, 3500, 250))[0].tolist()}. "
          f"Clusters near 1500 and 3000 would indicate 4-pole and 2-pole machines (different physics). Not stated in the file.")


def check_duplicates(b, X, a: Audit):
    cols = ["motor_id", "ts_raw"] + CLEAN
    full = int(pd.concat([b[["motor_id", "ts_raw"]], X], axis=1).duplicated().sum())
    same_vec = int(X.duplicated(keep=False).sum())
    a.add("D01", "Duplicate records", PASS if full == 0 else FAIL, f"fully identical (ignoring record id): {full}; rows with an identical 14-value sensor vector: {same_vec}")
    conflict = b.assign(**{c: X[c] for c in CLEAN})
    conflict = conflict[conflict.duplicated(CLEAN, keep=False)].groupby(CLEAN)[LABEL_RAW].nunique()
    a.add("D02", "Identical readings with different labels", PASS if (conflict > 1).sum() == 0 else WARN, f"{int((conflict > 1).sum())} identical sensor vectors carry more than one label (label noise)")


# ----------------------------------------------------------------------------------------------
# 5. Labels and targets
# ----------------------------------------------------------------------------------------------
def check_labels(b, X, fam, a: Audit):
    lab = b[LABEL_RAW]
    unknown = sorted(set(lab[lab != ""]) - set(CLASSES)); empty = int((lab == "").sum())
    a.add("L01", "Label values are exactly Normal / Alert / Danger", PASS if not unknown and not empty else FAIL, f"unknown values: {unknown or 'none'}; empty: {empty}")
    cnt = lab.value_counts().reindex(CLASSES).fillna(0).astype(int)
    exp = INSPECTED["labels"]
    a.add("L02", "Class distribution", PASS if cnt.to_dict() == exp else WARN,
          f"counts {cnt.to_dict()} ({(cnt / cnt.sum() * 100).round(1).to_dict()} %); earlier inspection {exp}; imbalance largest/smallest = {cnt.max() / max(cnt.min(), 1):.1f}:1")
    sev = lab.map({c: i for i, c in enumerate(CLASSES)})
    tab = pd.crosstab(b["motor_id"], lab).reindex(columns=CLASSES, fill_value=0)
    top3 = tab["Danger"].sort_values(ascending=False).head(3)
    a.add("L03", "How concentrated are Danger labels?", INFO if top3.sum() < 0.6 * max(tab["Danger"].sum(), 1) else WARN,
          f"motors with any Danger: {int((tab.Danger > 0).sum())} of {len(tab)}; top-3 motors hold {int(top3.sum())} of {int(tab.Danger.sum())} Danger records ({top3.to_dict()}). "
          f"Few motors carrying most Danger records makes any motor-grouped score highly dependent on which motors land in test.")
    famtab = pd.crosstab(fam, lab).reindex(columns=CLASSES, fill_value=0)
    famtab["Danger_%"] = (famtab["Danger"] / famtab.sum(axis=1) * 100).round(1)
    a.tables["labels_by_family"] = famtab.reset_index().rename(columns={"motor_id": "family"}).to_dict("records")
    a.add("L04", "Label mix differs by motor family", WARN if famtab["Danger_%"].max() - famtab["Danger_%"].min() > 5 else INFO,
          f"Danger % by family: {famtab['Danger_%'].to_dict()}. If families differ this much, 'family' is a hidden predictor and a random motor split does not test generalisation to a new family.")
    flips, impr, const = 0, 0, 0
    for m, g in b.assign(sev=sev, t=b.get("t")).groupby("motor_id"):
        s = g["sev"].to_numpy()
        flips += int((np.diff(s) != 0).sum()); impr += int((np.diff(s) < 0).sum()); const += int(len(set(s)) == 1)
    a.add("L05", "Label dynamics inside a motor (file order)", INFO,
          f"label changes: {flips}; improvements (e.g. Danger->Normal): {impr}; motors with one constant label: {const}. Improvements may mean repair, re-labelling or noise: ask the data owner.")


def check_targets(rows, b, X, a: Audit):
    header_cells = [c.strip() for r in rows[:2] for c in r if c.strip()]
    kw = re.compile(r"fail|event|repair|maint|breakdown|rul|horizon|next|within|7.?day|seven|remain", re.I)
    hits = [c for c in header_cells if kw.search(c)]
    a.tables["column_inventory"] = [{"column": c} for c in header_cells]
    a.add("G01", "Is a failure-event or 7-day failure target present in the file?", FAIL if not hits else INFO,
          (f"No header contains failure/event/repair/maintenance/horizon words. The ONLY target-like column is '{LABEL_RAW}' (current condition). "
           f"A '7-day failure' target does NOT exist in this file and cannot be created from it without confirmed failure events." if not hits else f"Candidate columns: {hits}. Inspect their definition and source."))
    if "t" in b:
        sev = b[LABEL_RAW].map({c: i for i, c in enumerate(CLASSES)})
        pos = cen = tot = 0
        for m, g in b.assign(sev=sev).sort_values(["motor_id", "t"]).groupby("motor_id"):
            t, s = g["t"].to_numpy(), g["sev"].to_numpy()
            for i in range(len(g)):
                tot += 1
                later = (t > t[i]) & (t <= t[i] + np.timedelta64(7, "D"))
                if not later.any():
                    cen += 1
                elif (s[later] == 2).any():
                    pos += 1
        a.add("G02", "PROXY only: 'a later Danger label within 7 days' (NOT a failure label)", INFO,
              f"{pos} of {tot} readings have a later Danger label within 7 days; {cen} have no later reading in the window (censored). "
              f"This counts future LABELS, not failures. Do not call it failure prediction unless Danger is shown to precede real failures.")


# ----------------------------------------------------------------------------------------------
# 6. Leakage and evaluation design
# ----------------------------------------------------------------------------------------------
def propose_grouped_split(df, seed=SEED):
    """Same procedure as notebook v1.0.0 section 11 (uses label counts only)."""
    motors = np.array(sorted(df["motor_id"].unique()))
    M = pd.crosstab(df["motor_id"], df["y"]).reindex(index=motors, columns=[0, 1, 2], fill_value=0).to_numpy()
    total = M.sum(0); glob = total / total.sum(); n = len(motors)
    n_tr, n_va = int(round(SPLIT_FRACTIONS[0] * n)), int(round(SPLIT_FRACTIONS[1] * n))
    rng = np.random.default_rng(seed); best = None; feasible = 0
    for _ in range(N_SPLIT_TRIALS):
        perm = rng.permutation(n); parts = [perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]]
        score, ok = 0.0, True
        for k, idx in enumerate(parts):
            c = M[idx].sum(0)
            if (c < MIN_CLASS_RECORDS).any():
                ok = False; break
            score += np.abs(c / c.sum() - glob).sum() + abs(c.sum() / total.sum() - SPLIT_FRACTIONS[k])
        if ok:
            feasible += 1
            if best is None or score < best[0]:
                best = (score, parts)
    if best is None:
        return None, 0
    return {nm: list(motors[idx]) for nm, idx in zip(["train", "val", "test"], best[1])}, feasible


def make_model_frame(b, X, fam):
    df = X.copy(); df.insert(0, "motor_id", b["motor_id"].values); df["family"] = fam.values
    df["y"] = b[LABEL_RAW].map({c: i for i, c in enumerate(CLASSES)}).values
    if "t" in b:
        df["t"] = b["t"].values
    return df


def check_leakage_and_split(b, X, fam, a: Audit, fast=False):
    try:
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import f1_score
        from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
        from sklearn.tree import DecisionTreeClassifier
    except ImportError:
        a.add("K00", "scikit-learn available", SKIP, "Install scikit-learn to run leakage screens.")
        return None
    if X.isna().any().any() or b[LABEL_RAW].isin(CLASSES).sum() != len(b):
        a.add("K00", "Leakage screens", SKIP, "Skipped because missing/invalid inputs or labels exist (fix those first; nothing was imputed).")
        return None
    df = make_model_frame(b, X, fam)
    y, g = df["y"].to_numpy(), df["motor_id"].to_numpy()
    # K01: file order / record number as a predictor (file is ordered motor then time)
    idx = np.arange(len(df)).reshape(-1, 1)
    p = cross_val_predict(DecisionTreeClassifier(max_depth=6, random_state=0), idx, y, cv=StratifiedKFold(5, shuffle=True, random_state=0))
    maj = max(np.bincount(y)) / len(y)
    a.add("K01", "Record number alone predicts the label (must stay out of features)", WARN if (p == y).mean() > maj + 0.03 else PASS,
          f"accuracy using ONLY file position: {(p == y).mean():.3f} vs majority baseline {maj:.3f}. A gain means row order carries motor/time information.")
    # K02: per-motor majority label = ceiling for memorising motor identity
    mm = df.groupby("motor_id")["y"].agg(lambda s: s.value_counts().idxmax())
    a.add("K02", "Motor identity alone as a predictor (memorisation ceiling)", INFO,
          f"always predicting each motor's own majority label scores {(df['motor_id'].map(mm) == df['y']).mean():.3f} (majority-class baseline {maj:.3f}). "
          f"This is why 'motor_id' is excluded and the split must keep a motor's rows together.")
    # K03: can sensors identify the motor / family? (fingerprint)
    Xv = df[FEATURES].to_numpy()
    rf = RandomForestClassifier(n_estimators=20 if fast else 150, random_state=0, n_jobs=-1)
    pm = cross_val_predict(rf, Xv, g, cv=StratifiedKFold(5, shuffle=True, random_state=0))
    pf = cross_val_predict(rf, Xv, df["family"].to_numpy(), cv=StratifiedKFold(5, shuffle=True, random_state=0))
    a.add("K03", "Sensors fingerprint the motor / family", WARN if (pm == g).mean() > 0.5 else INFO,
          f"motor identification accuracy from the 14 inputs: {(pm == g).mean():.3f} (chance {1 / df.motor_id.nunique():.3f}); family: {(pf == df['family']).mean():.3f} (chance {1 / df.family.nunique():.3f}). "
          f"High values mean readings from one motor are near-duplicates, so a random ROW split would leak.")
    # K04: label-rule probe on the WHOLE file with motor-grouped CV (diagnostic; no model selection involved)
    out = {}
    for name, cols in {"velocity only": [c for c in FEATURES if "velocity" in c], "gE only": [c for c in FEATURES if c.endswith("_ge")],
                       "temperature only": [c for c in FEATURES if "temp" in c], "operating only": ["rotational_speed_rpm", "load_percent"]}.items():
        best = 0
        for depth in (1, 2, 3):
            pr = cross_val_predict(DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=0), df[cols].to_numpy(), y, cv=GroupKFold(5), groups=g)
            best = max(best, f1_score(y, pr, average="macro"))
        out[name] = round(best, 3)
    a.add("K04", "Can a tiny rule on ONE sensor type reproduce the labels?", WARN if out["velocity only"] >= 0.9 or max(out.values()) >= 0.9 else INFO,
          f"best motor-grouped macro-F1 of a depth<=3 tree: {out}. Near 1.0 for one sensor type suggests the labels were DERIVED from that sensor (circularity). "
          f"Moderate values do not prove independence. Provenance from the data owner is still required.")
    # K05: operating columns as motor-type proxy
    within = df.groupby("motor_id")["rotational_speed_rpm"].std().median(); between = df.groupby("motor_id")["rotational_speed_rpm"].median().std()
    a.add("K05", "Operating conditions act like a motor-type ID", WARN if between > 5 * max(within, 1e-9) else INFO,
          f"rpm spread within a motor (median SD) {within:.1f} vs between motor medians {between:.1f}")
    # SPLIT
    split, feas = propose_grouped_split(df)
    if split is None:
        a.add("P01", "Reproduce notebook motor-grouped split", FAIL, "No partition satisfies the minimum-class rule with these labels.")
        return df, None
    comp = {}
    for s, ms in split.items():
        d = df[df.motor_id.isin(ms)]
        comp[s] = {"motors": len(ms), "records": len(d), **{c: int((d.y == i).sum()) for i, c in enumerate(CLASSES)},
                   "families": d.family.value_counts().to_dict(), "danger_in_top_motor_%": round(100 * d[d.y == 2].motor_id.value_counts().iloc[0] / max((d.y == 2).sum(), 1), 1)}
    a.tables["reproduced_split"] = comp
    a.add("P01", "Reproduced motor-grouped split (notebook v1.0.0 procedure, seed 42)", INFO,
          f"{feas} of {N_SPLIT_TRIALS} candidate partitions feasible. Compare this table with the one the notebook printed; they must match. {json.dumps(comp)}")
    fams_test = set(df[df.motor_id.isin(split['test'])].family)
    a.add("P02", "Split vs. research claim", WARN,
          f"A motor-grouped random split supports ONLY: 'does the model classify the recorded condition of UNSEEN MOTORS drawn from the SAME five families in the SAME Apr-Aug window'. "
          f"It does NOT support: (a) predicting FUTURE failures (no future-event target), (b) a NEW motor family or plant (test families {sorted(fams_test)} also appear in training), "
          f"(c) performance after time passes (train and test cover the same dates).")
    # leave-one-family-out table to quantify the family effect
    rows = []
    for f_ in ([] if fast else sorted(df.family.unique())):
        tr, te = df[df.family != f_], df[df.family == f_]
        r = RandomForestClassifier(n_estimators=150, random_state=0, n_jobs=-1, class_weight="balanced").fit(tr[FEATURES], tr.y)
        pr = r.predict(te[FEATURES])
        rows.append({"held_out_family": f_, "n": len(te), "danger_in_family": int((te.y == 2).sum()),
                     "macro_F1": round(float(f1_score(te.y, pr, average="macro", labels=[0, 1, 2], zero_division=0)), 3)})
    a.tables["leave_one_family_out_RF"] = rows
    a.add("P03", "Leave-one-FAMILY-out sensitivity (Random Forest, diagnostic)", SKIP if fast else INFO,
          "skipped in fast mode" if fast else
          f"{rows}. Large drops versus the motor-grouped score mean the headline number depends on the family being in training.")
    return df, split


# ----------------------------------------------------------------------------------------------
# 7. Bundle (training <-> inference contract)
# ----------------------------------------------------------------------------------------------
def numpy_forward(layers, mean, scale, X):
    z = (np.asarray(X, dtype="float64") - mean) / scale
    for W, bb, act in layers:
        z = z @ W + bb
        z = np.maximum(z, 0) if act == "relu" else (np.exp(z - z.max(1, keepdims=True)) / np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
    return z


def check_bundle(bundle_dir, raw_sha, df, split, a: Audit):
    d = Path(bundle_dir)
    need = ["preprocessing.json", "feature_schema.json", "class_mapping.json", "model_architecture.json", "weights.npz", "training_metadata.json", "parity_probe.json", "MANIFEST.json"]
    miss = [n for n in need if not (d / n).exists()]
    if miss:
        a.add("B01", "Bundle files present", FAIL, f"missing: {miss}")
        return
    J = lambda n: json.loads((d / n).read_text(encoding="utf-8"))
    man = J("MANIFEST.json")
    bad = [n for n, h in man["files"].items() if not (d / n).exists() or sha256_of(d / n) != h]
    a.add("B01", "Bundle files match MANIFEST checksums", PASS if not bad else FAIL, f"changed or missing: {bad or 'none'}")
    pre, sch, cls, arch, meta, par = J("preprocessing.json"), J("feature_schema.json"), J("class_mapping.json"), J("model_architecture.json"), J("training_metadata.json"), J("parity_probe.json")
    names = pre["feature_names"]
    a.add("B02", "Feature names and ORDER: training == bundle == schema", PASS if names == FEATURES and [f["name"] for f in sch["features"]] == FEATURES else FAIL,
          f"preprocessing={names == FEATURES}; schema={[f['name'] for f in sch['features']] == FEATURES}")
    units_ok = all(f["unit"] == SENSORS[r][1] for f, r in zip(sch["features"], SENSORS) if len(sch["features"]) == len(SENSORS))
    a.add("B03", "Units in feature_schema match the file's column units", PASS if units_ok else FAIL, "units compared column by column")
    a.add("B04", "Class order", PASS if [c["name"] for c in sorted(cls["classes"], key=lambda c: c["id"])] == CLASSES else FAIL, f"bundle classes {cls['classes']}")
    ds = (meta.get("dataset") or {})
    a.add("B05", "Model was trained on THIS data file", PASS if ds.get("sha256") == raw_sha else FAIL,
          f"metadata sha256 {str(ds.get('sha256'))[:16]}... vs file {raw_sha[:16]}...")
    a.add("B06", "Not a smoke-test bundle", PASS if not meta.get("fast_debug_run") else FAIL, f"fast_debug_run={meta.get('fast_debug_run')}")
    lp = meta.get("label_provenance") or {}
    todo = [k for k, v in lp.items() if k != "documented" and (v is None or (isinstance(v, str) and v.strip().upper().startswith("TODO")))]
    prov_flag = meta.get("provisional_label_provenance")
    if prov_flag is False and todo:
        a.add("B07", "Provenance flag is justified by evidence", FAIL, f"bundle says provenance is DOCUMENTED (provisional=False) but these fields are still empty/TODO: {todo}. The honesty gate was switched off without evidence.")
    elif prov_flag is False:
        a.add("B07", "Provenance flag is justified by evidence", WARN, f"Marked documented: {lp}. A program cannot verify this; check the evidence_reference document yourself.")
    else:
        a.add("B07", "Provenance flag is justified by evidence", PASS, "Bundle is stamped provisional (provenance not documented). This is the honest state if no written evidence exists.")
    # scaler fitted on TRAIN only? needs the reproduced split
    mean, scale = np.array(pre["mean"]), np.array(pre["scale"])
    if df is not None and split is not None:
        tr = df[df.motor_id.isin(split["train"])][FEATURES].to_numpy(); al = df[FEATURES].to_numpy()
        e_tr = float(np.abs(mean - tr.mean(0)).max() / (np.abs(tr.mean(0)).max() + 1e-12)); e_al = float(np.abs(mean - al.mean(0)).max() / (np.abs(al.mean(0)).max() + 1e-12))
        s_tr = float(np.abs(scale - tr.std(0, ddof=0)).max()); s_al = float(np.abs(scale - al.std(0, ddof=0)).max())
        ok_tr = e_tr < 1e-6 and s_tr < 1e-6
        ok_al = e_al < 1e-6 and s_al < 1e-6
        a.add("B08", "Scaler parameters were learned from TRAINING motors only", PASS if ok_tr and not ok_al else FAIL if ok_al else WARN,
              f"max relative mean diff vs reproduced-train {e_tr:.2e} / vs all-data {e_al:.2e}; max scale diff {s_tr:.2e} / {s_al:.2e}. "
              + ("Matches training motors only." if ok_tr and not ok_al else "MATCHES ALL DATA: test/validation information leaked into scaling." if ok_al else
                 "Matches neither: the reproduced split or seed differs from the one actually used (compare split tables)."))
        sm = (meta.get("split") or {}).get("summary") or {}
        rep = {s: {"motors": len(split[s]), "records": int(df.motor_id.isin(split[s]).sum())} for s in split}
        a.add("B09", "Split recorded in the bundle equals the reproduced split", PASS if sm == rep else WARN, f"bundle {sm} vs reproduced {rep}")
    # inference parity on the artificial probes with an independent forward pass
    z = np.load(d / "weights.npz", allow_pickle=False)
    layers = [(z[f"W{i}"], z[f"b{i}"], s["activation"]) for i, s in enumerate(arch["layers"])]
    worst = 0.0
    for p in par["probes"]:
        got = numpy_forward(layers, mean, scale, np.array([[p["readings"][n] for n in names]]))[0]
        worst = max(worst, float(np.abs(got - np.array([p["expected_probabilities"][c] for c in CLASSES])).max()))
    a.add("B10", "Independent NumPy forward pass reproduces the notebook's probe outputs", PASS if worst < 1e-5 else FAIL, f"max abs difference {worst:.2e} over {len(par['probes'])} probes (artificial inputs: not real rows)")


# ----------------------------------------------------------------------------------------------
# 8. Cleaned copy equals the original (nothing altered)
# ----------------------------------------------------------------------------------------------
def check_clean_copy(path, b, X, a: Audit):
    c = pd.read_csv(path, dtype=str, keep_default_na=False)
    exp = ["record_id", "motor_id", "measured_at"] + CLEAN + ["condition_label"]
    a.add("C01", "Clean copy has the expected columns", PASS if list(c.columns) == exp else FAIL, f"columns {list(c.columns)}")
    a.add("C02", "No row dropped or added", PASS if len(c) == len(b) and set(c.record_id) == set(b.record_id) else FAIL, f"clean {len(c)} vs raw {len(b)}")
    m = c.set_index("record_id").join(b.set_index("record_id")[["motor_id", LABEL_RAW]].rename(columns={"motor_id": "mid_raw", LABEL_RAW: "lab_raw"}), how="inner")
    Xi = X.set_axis(b["record_id"].values)
    num_diff = int((~np.isclose(m[CLEAN].astype(float).to_numpy(), Xi.loc[m.index, CLEAN].to_numpy(), rtol=0, atol=0)).sum()) if len(m) else -1
    a.add("C03", "Every sensor value is identical to the original (no cleaning, rounding or imputation)", PASS if num_diff == 0 else FAIL, f"differing cells: {num_diff}")
    a.add("C04", "Motor IDs and labels identical to the original", PASS if (m.motor_id == m.mid_raw).all() and (m.condition_label == m.lab_raw).all() else FAIL, "compared row by row")


# ----------------------------------------------------------------------------------------------
def guarded(a: Audit, stage, fn, *args, **kw):
    """Run one stage; if it crashes on malformed input, record a FAIL instead of stopping the whole audit."""
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001 - deliberate: a crash on bad input is itself a finding
        a.add(stage, f"Stage '{fn.__name__}' could not complete", FAIL, f"{type(exc).__name__}: {exc}. The input is malformed in a way this stage cannot read; fix it and re-run.")
        return None


def run(data, out, clean_copy=None, bundle=None, fast=False):
    a = Audit()
    sha_before = sha256_of(data)
    rows, sha = read_text_grid(data, a)
    b = split_header_and_body(rows, a)
    if b is None:
        return a, sha_before
    fam = check_ids(b, a)
    fam = fam.set_axis(b.index)
    b = check_timestamps(b, a)
    X = check_numbers(b, a)
    guarded(a, "U99", check_units, b, X, fam, a)
    guarded(a, "D99", check_duplicates, b, X, a)
    guarded(a, "L99", check_labels, b, X, fam, a)
    guarded(a, "G99", check_targets, rows, b, X, a)
    df, split = (guarded(a, "K99", check_leakage_and_split, b, X, fam, a, fast) or (None, None))
    if clean_copy:
        guarded(a, "C99", check_clean_copy, clean_copy, b, X, a)
    if bundle:
        guarded(a, "B99", check_bundle, bundle, sha, df, split, a)
    a.add("Z01", "Input file unchanged by this audit", PASS if sha256_of(data) == sha_before else FAIL, f"SHA-256 before = after = {sha_before[:16]}...")
    a.add("Z02", "Label provenance", WARN,
          "UNVERIFIED by any program. Who/what assigned Normal/Alert/Danger, from which data, and whether it was confirmed by maintenance records "
          "must come from written evidence supplied by the data owner. Until then treat every label as unverified.")
    return a, sha_before


def verdict(a: Audit):
    fails = [f for f in a.findings if f["status"] == FAIL]
    if fails:
        return "NOT VALIDATED: " + str(len(fails)) + " check(s) FAILED (" + ", ".join(f["id"] for f in fails) + ")"
    return "STRUCTURE CHECKS PASSED, but LABEL PROVENANCE IS UNVERIFIED. This is not a declaration that the dataset is valid."


def write_reports(a: Audit, out: Path, data):
    out.mkdir(parents=True, exist_ok=True)
    meta = {"audit_version": AUDIT_VERSION, "run_at": datetime.now().isoformat(timespec="seconds"), "input": str(data), "verdict": verdict(a)}
    (out / "audit_report.json").write_text(json.dumps({**meta, "findings": a.findings, "tables": a.tables}, indent=2, default=str), encoding="utf-8")
    L = [f"# Dataset audit report", f"- Run: {meta['run_at']} | audit v{AUDIT_VERSION} | input: `{Path(data).name}`", f"- **Verdict: {meta['verdict']}**", "",
         "| ID | Status | Check | Detail |", "|---|---|---|---|"]
    for f in a.findings:
        L.append(f"| {f['id']} | **{f['status']}** | {f['title']} | {str(f['detail']).replace('|', '/')} |")
    for k, v in a.tables.items():
        L += ["", f"### {k}", "```", json.dumps(v, indent=1, default=str)[:4000], "```"]
    (out / "audit_report.md").write_text("\n".join(L), encoding="utf-8")
    return meta


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True, help="raw master CSV (read-only)")
    ap.add_argument("--clean-copy", help="motor_clean_working_copy.csv written by the notebook")
    ap.add_argument("--bundle", help="folder of the exported model bundle")
    ap.add_argument("--fast", action="store_true", help="smaller models for the leakage screens (smoke tests only)")
    ap.add_argument("--out", default="audit_outputs", help="where reports are written (never next to the original unless you choose so)")
    args = ap.parse_args(argv)
    if not Path(args.data).exists():
        print("File not found:", args.data); return 2
    if Path(args.out).resolve() == Path(args.data).resolve().parent:
        print("Refusing to write reports into the folder that holds the original. Choose another --out."); return 2
    a, _ = run(args.data, Path(args.out), args.clean_copy, args.bundle, args.fast)
    meta = write_reports(a, Path(args.out), args.data)
    sym = {FAIL: "FAIL", WARN: "WARN", PASS: "pass", INFO: "info", SKIP: "skip"}
    for f in a.findings:
        print(f"[{sym[f['status']]:4}] {f['id']}  {f['title']}")
    print("\n" + meta["verdict"]); print("Reports:", Path(args.out).resolve())
    return 1 if any(f["status"] == FAIL for f in a.findings) else 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile notebook_leakage_checks.py
# ======================================================================================================
# IN-NOTEBOOK LEAKAGE CHECKS  (paste into a NEW cell right AFTER section 12 "Fit preprocessing", i.e. after
# `scaler = StandardScaler().fit(X_train)` and BEFORE section 13 trains the ANN. It never touches the test labels
# for fitting or selection, and it changes nothing in your variables.)
#
# Needs these notebook variables: df, FEATURES, EXCLUDED, CLASS_NAMES, split_motors, train, val, test,
# X_train, y_train, X_val_s, y_val, scaler, MIN_CLASS_RECORDS
# ======================================================================================================
import numpy as np, pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


EXPECTED_FEATURES = ["de_h_velocity_mm_s", "de_h_ge", "de_h_temp_c", "de_v_velocity_mm_s", "de_v_ge", "de_v_temp_c",
                     "nde_h_velocity_mm_s", "nde_h_ge", "nde_h_temp_c", "nde_v_velocity_mm_s", "nde_v_ge", "nde_v_temp_c",
                     "rotational_speed_rpm", "load_percent"]      # the 14-input contract shared with the backend


def run_notebook_leakage_checks(ns, n_perm=5, seed=0):
    g = lambda k: ns[k]
    df, FEATURES, EXCLUDED = g("df"), g("FEATURES"), g("EXCLUDED")
    split_motors, train, val, test = g("split_motors"), g("train"), g("val"), g("test")
    X_train, y_train, X_val_s, y_val, scaler = g("X_train"), g("y_train"), g("X_val_s"), g("y_val"), g("scaler")
    rows = []
    add = lambda cid, title, ok, detail, warn=False: rows.append({"check": cid, "result": "PASS" if ok else ("WARN" if warn else "FAIL"), "title": title, "detail": detail})

    # N1 scaler fitted on the TRAINING rows only
    d_tr = float(np.abs(scaler.mean_ - X_train.mean(0)).max()); d_all = float(np.abs(scaler.mean_ - df[FEATURES].to_numpy().mean(0)).max())
    add("N1", "Scaler mean equals the TRAINING mean and not the all-data mean", d_tr < 1e-9 and d_all > 1e-9,
        f"max |scaler.mean_ - train mean| = {d_tr:.2e}; max |scaler.mean_ - all-data mean| = {d_all:.2e}; n_samples_seen_ = {int(scaler.n_samples_seen_)} (train rows = {len(X_train)})")
    # N2 split integrity
    s = {k: set(v) for k, v in split_motors.items()}
    disjoint = s["train"].isdisjoint(s["val"]) and s["train"].isdisjoint(s["test"]) and s["val"].isdisjoint(s["test"])
    complete = set().union(*s.values()) == set(df.motor_id.unique())
    add("N2", "Every motor is in exactly one split", disjoint and complete, f"disjoint={disjoint}, complete={complete}")
    minrec = ns.get("MIN_CLASS_RECORDS", 10)
    cnt = {k: np.bincount(d["y"], minlength=3) for k, d in {"train": train, "val": val, "test": test}.items()}
    add("N3", f"Each split has >= {minrec} records of every class", all((c >= minrec).all() for c in cnt.values()), f"class counts per split (Normal, Alert, Danger): { {k: c.tolist() for k, c in cnt.items()} }")
    # N4 forbidden inputs
    bad = [f for f in FEATURES if f in EXCLUDED or f in ("y", "split", "family", "t")]
    extra, missing = [f for f in FEATURES if f not in EXPECTED_FEATURES], [f for f in EXPECTED_FEATURES if f not in FEATURES]
    add("N4", "Model inputs are exactly the 14-input contract (no identifier, time, label or split column)",
        not bad and not extra and not missing and list(FEATURES) == EXPECTED_FEATURES,
        f"{len(FEATURES)} inputs; forbidden present: {bad or 'none'}; extra: {extra or 'none'}; missing: {missing or 'none'}; same order as the backend contract: {list(FEATURES) == EXPECTED_FEATURES}")
    # N5 label-permutation test: a pipeline with no leakage collapses to chance when labels are shuffled
    rng = np.random.default_rng(seed); scores = []
    Xs = scaler.transform(X_train)
    for _ in range(n_perm):
        yp = rng.permutation(y_train)
        m = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xs, yp)
        scores.append(f1_score(y_val, m.predict(X_val_s), average="macro", labels=[0, 1, 2], zero_division=0))
    real = f1_score(y_val, LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xs, y_train).predict(X_val_s), average="macro", labels=[0, 1, 2], zero_division=0)
    add("N5", "Shuffled-label control collapses to chance on validation motors", float(np.mean(scores)) < 0.45,
        f"macro-F1 with shuffled training labels: mean {np.mean(scores):.3f} (max {np.max(scores):.3f}); with real labels {real:.3f}. A shuffled-label score well above ~0.33 means information reaches the model by a route other than the sensors-to-label relationship.")
    # N6 position-only baselines (record order / calendar time carry motor and period information)
    base = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_train), 1)), y_train)
    dummy = f1_score(y_val, base.predict(np.zeros((len(y_val), 1))), average="macro", labels=[0, 1, 2], zero_division=0)
    pos = lambda d: np.c_[d["measured_at"].astype("int64") // 10**9] if "measured_at" in d else np.arange(len(d)).reshape(-1, 1)
    pm = RandomForestClassifier(n_estimators=100, random_state=0).fit(pos(train), y_train)
    f_pos = f1_score(y_val, pm.predict(pos(val)), average="macro", labels=[0, 1, 2], zero_division=0)
    add("N6", "Calendar time alone does not beat the do-nothing baseline on unseen motors", f_pos <= dummy + 0.10, f"validation macro-F1: time-only {f_pos:.3f} vs Dummy {dummy:.3f}", warn=True)
    # N7 operating columns alone
    ops = ["rotational_speed_rpm", "load_percent"]
    om = RandomForestClassifier(n_estimators=100, random_state=0, class_weight="balanced").fit(train[ops], y_train)
    f_ops = f1_score(y_val, om.predict(val[ops]), average="macro", labels=[0, 1, 2], zero_division=0)
    add("N7", "Speed + load alone are not a strong predictor", f_ops <= dummy + 0.15, f"validation macro-F1 from rpm+load only {f_ops:.3f} vs Dummy {dummy:.3f}. High = they identify motor type/family, not condition.", warn=True)
    # N8 family coverage and calendar overlap (limits of the claim)
    fam = df.motor_id.str.replace(r"\d+$", "", regex=True)
    fam_split = pd.crosstab(fam.values, df["split"].values) if "split" in df else None
    t_rng = {k: (str(d["measured_at"].min()), str(d["measured_at"].max())) for k, d in {"train": train, "val": val, "test": test}.items()}
    add("N8", "Claim limits: test motors come from families and dates already seen in training", False,
        f"motors per family per split: {fam_split.to_dict() if fam_split is not None else 'n/a'}; date ranges {t_rng}. Report the score as 'unseen motors of known families in the same period', not as future or new-plant performance.", warn=True)
    # N9 any single input that almost reproduces the label on unseen motors (label-derived column or label-rule)
    from sklearn.tree import DecisionTreeClassifier
    top = []
    for i, f in enumerate(FEATURES):
        t = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=0).fit(train[[f]], y_train)
        top.append((round(float(f1_score(y_val, t.predict(val[[f]]), average="macro", labels=[0, 1, 2], zero_division=0)), 3), f))
    top.sort(reverse=True)
    add("N9", "No single input almost reproduces the label on unseen motors", top[0][0] < 0.90, f"best single-input depth-2 tree validation macro-F1: {top[:3]}. "
        "Near 1.0 suggests the label was derived from that column (see audit K04) or the column is label-derived.", warn=True)
    out = pd.DataFrame(rows)
    print(out.to_string(index=False, max_colwidth=160))
    hard = out[out.result == "FAIL"]
    assert hard.empty, "LEAKAGE CHECK FAILED: " + ", ".join(hard.check)
    return out


if "df" in globals() and "scaler" in globals():          # runs automatically when pasted into the notebook
    LEAKAGE_CHECKS = run_notebook_leakage_checks(globals())


In [ ]:
%%writefile tests/conftest.py
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))


In [ ]:
%%writefile tests/fixture_builder.py
"""Builds SYNTHETIC files with the same layout as the real raw master CSV (two header rows, 18 columns).
They exist ONLY to test the audit tools. They contain no real measurements and prove nothing about your data."""
import csv, io
from datetime import datetime, timedelta
import numpy as np

GROUPS = ["Record ID", "Motor ID", "Measurement timestamps", "Drive end Header", "", "", "Drive end Vrtex", "", "",
          "Non Drive end Header", "", "", "Non Drive end Vrtex", "", "", "", "", ""]
SUB = ["", "", "", "DE_H_mm/s", "DE_H_gE", "DE_H_temp_C", "DE_V_mm/s", "DE_V_gE", "DE_V_temp_C", "NDE_H_mm/s", "NDE_H_gE", "NDE_H_temp_C",
       "NDE_V_mm/s", "NDE_V_gE", "NDE_V_temp_C", "rotational_speed_rpm", "load_percent", "condition_label"]
FAMS = ["CIPR", "CIPS", "MRP", "MSA", "SDP"]


def build_rows(rule_derived=False, seed=0, n_per_motor=45):
    """rule_derived=True makes labels a pure function of DE_H velocity (a 'circular' label set)."""
    rng = np.random.default_rng(seed)
    rows, rid = [], 0
    for fi, fam in enumerate(FAMS):
        for k in range(1, 9):
            motor = f"{fam}{k:02d}"
            base_v = 0.7 + 0.15 * fi + rng.normal(0, 0.05)
            rpm0 = 1460 if fi != 3 else 2960
            risk = rng.random()                                 # some motors deteriorate
            t0 = datetime(2025, 4, 1, 7 + k % 3, (5 + 7 * k) % 60)
            for i in range(n_per_motor):
                drift = (i / n_per_motor) * (2.5 if risk > 0.7 else 0.3)
                v = base_v + drift + rng.normal(0, 0.12)
                vel = [max(v + rng.normal(0, .05), .1) for _ in range(4)]
                ge = [max(0.3 + 0.2 * drift + rng.normal(0, 0.05), .05) for _ in range(4)]
                tp = [48 + 4 * drift + rng.normal(0, 1.5) for _ in range(4)]
                if rule_derived:
                    lab = "Danger" if vel[0] > 2.5 else "Alert" if vel[0] > 1.5 else "Normal"
                else:
                    s = 0.5 * vel[0] + 2 * ge[0] + 0.05 * tp[0] + rng.normal(0, 1.2)       # noisy: not a clean rule
                    lab = "Danger" if s > 6.6 else "Alert" if s > 5.9 else "Normal"
                ts = t0 + timedelta(days=3 * i, minutes=int(rng.integers(0, 40)))
                rid += 1
                rows.append([f"REC{rid:06d}", motor, f"{ts.month}/{ts.day}/{ts.year} {ts.hour}:{ts.minute:02d}",
                             f"{vel[0]:.2f}", f"{ge[0]:.3f}", f"{tp[0]:.1f}", f"{vel[1]:.2f}", f"{ge[1]:.3f}", f"{tp[1]:.1f}",
                             f"{vel[2]:.2f}", f"{ge[2]:.3f}", f"{tp[2]:.1f}", f"{vel[3]:.2f}", f"{ge[3]:.3f}", f"{tp[3]:.1f}",
                             str(int(rpm0 + rng.integers(-8, 8))), f"{rng.uniform(55, 85):.1f}", lab])
    return rows


def to_csv(rows):
    buf = io.StringIO(newline="")
    w = csv.writer(buf, lineterminator="\r\n")
    w.writerow(GROUPS); w.writerow(SUB)
    for r in rows:
        w.writerow(r)
    return "﻿" + buf.getvalue()


def write(path, rows):
    with open(path, "w", encoding="utf-8", newline="") as fh:
        fh.write(to_csv(rows))
    return path


def plant_defects(rows):
    """Return (new_rows, expected) where expected maps audit-check id -> status that MUST appear."""
    r = [list(x) for x in rows]
    r[9][0] = r[8][0]                                    # I01 duplicate record id
    r[20][3 + 1] = ""                                    # N01 empty measurement
    r[21][3 + 2] = "N/A"                                 # N02 text in numeric column
    r[22][5] = "-999"                                    # N03 sentinel + N04 negative
    r[30][2] = "13/02/2025 08:00"                        # T03 day-first only (mixed conventions)
    r[31][2] = "31/31/2025 08:00"                        # T01 unparseable
    r[50][2], r[51][2] = r[51][2], r[50][2]              # T07 out of order within a motor
    r[60][2] = r[59][2]                                  # T05 same motor + timestamp
    r[70][1] = "cipr01 "                                 # M01 spelling variant of CIPR01 (surrounding space is stripped)
    r[80][17] = "Critical"                               # L01 unknown label
    for row in r[135:180]:                               # U01 one motor reported in Fahrenheit
        for j in (5, 8, 11, 14):
            row[j] = f"{float(row[j]) * 1.8 + 32:.1f}"
    r.append(list(r[100]) )                              # D01 an exact repeat of another reading (new record id below)
    r[-1][0] = "REC999999"
    return r, {"I01": "FAIL", "N01": "FAIL", "N02": "FAIL", "N03": "WARN", "N04": "FAIL", "T03": "FAIL", "T01": "FAIL",
               "T07": "FAIL", "T05": "FAIL", "M01": "FAIL", "L01": "FAIL", "U01": "WARN", "D01": "FAIL"}


In [ ]:
%%writefile tests/test_dataset_contract.py
"""Automated data-quality tests.

Part A (always runs): proves the audit tool itself detects planted defects, using SYNTHETIC fixtures.
Part B (runs only when you point it at your real files): the actual contract checks.
    PM_DATA_PATH=/content/motor_predictive_maintenance_master.csv   raw master file
    PM_CLEAN_COPY=/content/outputs/motor_clean_working_copy.csv     optional
    PM_BUNDLE_PATH=/content/outputs/pm_condition_ann_v1.0.0         optional
Without those variables, Part B tests are reported as SKIPPED, never as passed.

Run:  python -m pytest tests -v -rsx
"""
import hashlib, json, os, shutil
from pathlib import Path

import numpy as np
import pytest

import fixture_builder as fb
import pm_data_audit as P

REAL = os.environ.get("PM_DATA_PATH")
REAL_CLEAN = os.environ.get("PM_CLEAN_COPY")
REAL_BUNDLE = os.environ.get("PM_BUNDLE_PATH")


def status(a):
    return {f["id"]: f["status"] for f in a.findings}


def audit(path, tmp, **kw):
    a, _ = P.run(str(path), Path(tmp) / "out", fast=True, **kw)      # fast: tool tests do not need full-size models
    return a


# ----------------------------------------------------------------------------- Part A: the tool works
@pytest.fixture(scope="module")
def clean_file(tmp_path_factory):
    d = tmp_path_factory.mktemp("clean")
    return fb.write(d / "raw.csv", fb.build_rows()), d


def test_clean_fixture_has_no_unexpected_failure(clean_file):
    p, d = clean_file
    st = status(audit(p, d))
    assert [k for k, v in st.items() if v == P.FAIL] == ["G01"], "only the missing failure-event target may FAIL on the fixture"


DEFECTS, EXPECTED = fb.plant_defects(fb.build_rows())


@pytest.fixture(scope="module")
def defect_status(tmp_path_factory):
    d = tmp_path_factory.mktemp("bad")
    return status(audit(fb.write(d / "bad.csv", DEFECTS), d))


@pytest.mark.parametrize("check_id,expected", sorted(EXPECTED.items()))
def test_planted_defect_is_detected(defect_status, check_id, expected):
    assert defect_status[check_id] == expected, f"{check_id} should be {expected}"


def test_fahrenheit_motor_is_named(tmp_path):
    rows, _ = fb.plant_defects(fb.build_rows())
    a = audit(fb.write(tmp_path / "f.csv", rows), tmp_path)
    detail = next(f["detail"] for f in a.findings if f["id"] == "U01")
    assert "CIPR04" in detail and "temp" in detail


def test_rule_derived_labels_are_flagged(tmp_path):
    a = audit(fb.write(tmp_path / "r.csv", fb.build_rows(rule_derived=True)), tmp_path)
    assert status(a)["K04"] == P.WARN


def test_audit_never_modifies_input(tmp_path):
    p = fb.write(tmp_path / "x.csv", fb.build_rows())
    before = hashlib.sha256(p.read_bytes()).hexdigest()
    a = audit(p, tmp_path / "o")
    assert hashlib.sha256(p.read_bytes()).hexdigest() == before and status(a)["Z01"] == P.PASS


def test_refuses_to_write_next_to_original(tmp_path):
    p = fb.write(tmp_path / "x.csv", fb.build_rows())
    assert P.main(["--data", str(p), "--out", str(tmp_path)]) == 2


def test_verdict_never_says_valid(clean_file):
    p, d = clean_file
    v = P.verdict(audit(p, d)).lower()
    assert "unverified" in v or "not validated" in v


def test_provenance_is_always_reported_unverified(clean_file):
    p, d = clean_file
    assert status(audit(p, d))["Z02"] == P.WARN


def test_target_inventory_finds_a_real_event_column(tmp_path):
    rows = fb.build_rows()
    text = fb.to_csv(rows).replace("condition_label", "condition_label").replace("Motor ID", "Motor ID")
    # add a header cell that looks like a failure target to prove the detector can say "present"
    lines = text.split("\r\n"); lines[0] = lines[0].rstrip(",") + ""
    p = tmp_path / "t.csv"; p.write_text(text.replace("Measurement timestamps", "Measurement timestamps"), encoding="utf-8")
    a = audit(p, tmp_path)
    assert status(a)["G01"] == P.FAIL                      # fixture has no failure column
    kw_text = text.replace("Record ID", "Failure within 7 days flag")
    p2 = tmp_path / "t2.csv"; p2.write_text(kw_text, encoding="utf-8")
    assert status(audit(p2, tmp_path))["G01"] != P.FAIL


# ----- bundle checks on synthetic bundles (good and deliberately broken)
def make_bundle(root, df, split, raw_sha, *, scaler_on="train", order_swap=False, documented=False, todo=False):
    root.mkdir(parents=True, exist_ok=True)
    feats = list(P.FEATURES)
    tr = df[df.motor_id.isin(split["train"])][feats].to_numpy() if scaler_on == "train" else df[feats].to_numpy()
    mean, scale = tr.mean(0), tr.std(0, ddof=0)
    rng = np.random.default_rng(1)
    W0, b0, W1, b1 = rng.normal(0, .3, (14, 8)), rng.normal(0, .1, 8), rng.normal(0, .3, (8, 3)), rng.normal(0, .1, 3)
    np.savez(root / "weights.npz", W0=W0, b0=b0, W1=W1, b1=b1)
    layers = [(W0, b0, "relu"), (W1, b1, "softmax")]
    names = feats[::-1] if order_swap else feats
    probes = []
    for nm, vec in {"mean": mean, "plus1": mean + scale}.items():
        probes.append({"name": nm, "readings": dict(zip(feats, vec)), "expected_probabilities": dict(zip(P.CLASSES, P.numpy_forward(layers, mean, scale, vec.reshape(1, -1))[0]))})
    w = lambda n, o: (root / n).write_text(json.dumps(o), encoding="utf-8")
    w("preprocessing.json", {"type": "StandardScaler", "feature_names": names, "mean": list(mean), "scale": list(scale)})
    w("feature_schema.json", {"features": [{"name": n, "unit": dict((v[0], v[1]) for v in P.SENSORS.values())[n]} for n in names]})
    w("class_mapping.json", {"classes": [{"id": i, "name": c} for i, c in enumerate(P.CLASSES)]})
    w("model_architecture.json", {"layers": [{"units": 8, "activation": "relu"}, {"units": 3, "activation": "softmax"}]})
    lp = {"documented": documented, "assigned_by": "TODO: who" if todo else "maintenance engineer", "rule_or_standard": "TODO" if todo else "human judgement",
          "assigned_from_same_sensor_columns": None if todo else False, "evidence_reference": "TODO" if todo else "email 2026-10-01"}
    summ = {s: {"motors": len(split[s]), "records": int(df.motor_id.isin(split[s]).sum())} for s in split}
    w("training_metadata.json", {"bundle_version": "v1", "provisional_label_provenance": not documented, "label_provenance": lp, "fast_debug_run": False,
                                 "dataset": {"sha256": raw_sha}, "split": {"summary": summ}})
    w("parity_probe.json", {"tolerance": 1e-5, "probes": probes})
    files = sorted(p.name for p in root.iterdir() if p.is_file())
    w("MANIFEST.json", {"files": {n: P.sha256_of(root / n) for n in files}})
    return root


@pytest.fixture(scope="module")
def prepared(tmp_path_factory):
    d = tmp_path_factory.mktemp("b")
    p = fb.write(d / "raw.csv", fb.build_rows())
    a = audit(p, d)
    import pandas as pd
    # rebuild the model frame exactly as the audit does
    rows, _ = None, None
    from pm_data_audit import read_text_grid, split_header_and_body, check_ids, check_timestamps, check_numbers, make_model_frame, propose_grouped_split, Audit
    t = Audit(); g, sha = read_text_grid(p, t); b = split_header_and_body(g, t); fam = check_ids(b, t).set_axis(b.index); b = check_timestamps(b, t); X = check_numbers(b, t)
    df = make_model_frame(b, X, fam); split, _ = propose_grouped_split(df)
    return d, p, df, split, sha


def bundle_status(prepared, name, **kw):
    d, p, df, split, sha = prepared
    root = make_bundle(d / name, df, split, kw.pop("sha", sha), **kw)
    return status(audit(p, d, bundle=str(root)))


def test_good_bundle_passes(prepared):
    st = bundle_status(prepared, "good")
    assert {k: v for k, v in st.items() if k.startswith("B")} == {k: P.PASS for k in ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09", "B10"]}


def test_scaler_fitted_on_all_data_is_caught(prepared):
    assert bundle_status(prepared, "leaky", scaler_on="all")["B08"] == P.FAIL


def test_feature_order_swap_is_caught(prepared):
    assert bundle_status(prepared, "swap", order_swap=True)["B02"] == P.FAIL


def test_model_trained_on_another_file_is_caught(prepared):
    assert bundle_status(prepared, "other", sha="0" * 64)["B05"] == P.FAIL


def test_documented_without_evidence_is_caught(prepared):
    assert bundle_status(prepared, "nodoc", documented=True, todo=True)["B07"] == P.FAIL


def test_tampered_bundle_file_is_caught(prepared):
    d, p, df, split, sha = prepared
    root = make_bundle(d / "tamper", df, split, sha)
    (root / "class_mapping.json").write_text("{}", encoding="utf-8")
    assert status(audit(p, d, bundle=str(root)))["B01"] == P.FAIL


def test_modified_clean_copy_is_caught(prepared, tmp_path):
    d, p, df, split, sha = prepared
    import pandas as pd
    clean = df.copy()
    c = pd.read_csv(p, header=None, dtype=str, skiprows=2)
    out = pd.DataFrame({"record_id": c[0], "motor_id": c[1], "measured_at": c[2]})
    for i, n in enumerate(P.CLEAN):
        out[n] = c[3 + i]
    out["condition_label"] = c[17]
    good = tmp_path / "good.csv"; out.to_csv(good, index=False)
    assert status(audit(p, d, clean_copy=str(good)))["C03"] == P.PASS
    out.loc[5, "de_h_ge"] = str(float(out.loc[5, "de_h_ge"]) + 0.001)
    bad = tmp_path / "bad.csv"; out.to_csv(bad, index=False)
    assert status(audit(p, d, clean_copy=str(bad)))["C03"] == P.FAIL


# ----------------------------------------------------------------------------- Part B: YOUR real files
needs_data = pytest.mark.skipif(not REAL, reason="set PM_DATA_PATH to run checks on the real dataset")


@pytest.fixture(scope="module")
def real(tmp_path_factory):
    d = tmp_path_factory.mktemp("real")
    a, _ = P.run(REAL, d / "o", REAL_CLEAN, REAL_BUNDLE)
    return status(a), a


HARD = ["S01", "S02", "S03", "I01", "M01", "T01", "T03", "T05", "T07", "N01", "N02", "N04", "D01", "L01", "Z01"]


@needs_data
@pytest.mark.parametrize("check_id", HARD)
def test_real_data_hard_contract(real, check_id):
    assert real[0][check_id] != P.FAIL, next(f["detail"] for f in real[1].findings if f["id"] == check_id)


@needs_data
@pytest.mark.xfail(reason="No failure-event / 7-day target exists in the file (see G01). Remove this marker only when a documented target is added.", strict=False)
def test_real_data_has_a_documented_failure_target(real):
    assert real[0]["G01"] != P.FAIL


@needs_data
def test_real_data_is_the_inspected_file(real):
    assert real[0]["F03"] == P.PASS, "the file differs from the one inspected earlier"


@pytest.mark.skipif(not (REAL and REAL_CLEAN), reason="set PM_CLEAN_COPY to compare the cleaned copy with the original")
@pytest.mark.parametrize("check_id", ["C01", "C02", "C03", "C04"])
def test_real_clean_copy_matches_original(real, check_id):
    assert real[0][check_id] == P.PASS


@pytest.mark.skipif(not (REAL and REAL_BUNDLE), reason="set PM_BUNDLE_PATH to audit the exported model bundle")
@pytest.mark.parametrize("check_id", ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B10"])
def test_real_bundle_contract(real, check_id):
    assert real[0][check_id] != P.FAIL, next(f["detail"] for f in real[1].findings if f["id"] == check_id)


In [ ]:
%%writefile tests/test_notebook_checks.py
"""Tests the in-notebook leakage block on SYNTHETIC data (no TensorFlow needed)."""
import numpy as np, pytest
from sklearn.preprocessing import StandardScaler
import fixture_builder as fb
import pm_data_audit as P
import notebook_leakage_checks as N


def namespace(tmp_path, *, fit_on="train", leak_label_feature=False, rule=False):
    p = fb.write(tmp_path / "r.csv", fb.build_rows(rule_derived=rule))
    t = P.Audit(); g, sha = P.read_text_grid(p, t); b = P.split_header_and_body(g, t)
    fam = P.check_ids(b, t).set_axis(b.index); b = P.check_timestamps(b, t); X = P.check_numbers(b, t)
    df = P.make_model_frame(b, X, fam).rename(columns={"t": "measured_at"})
    split, _ = P.propose_grouped_split(df)
    df["split"] = df.motor_id.map({m: s for s, ms in split.items() for m in ms})
    train, val, test = (df[df.split == s].copy() for s in ["train", "val", "test"])
    feats = list(P.FEATURES)
    if leak_label_feature:
        for d in (df, train, val, test):
            d["leak"] = d["y"] * 1.0
        feats = feats + ["leak"]
    Xtr = train[feats].to_numpy(float)
    scaler = StandardScaler().fit(Xtr if fit_on == "train" else df[feats].to_numpy(float))
    return dict(df=df, FEATURES=feats, EXCLUDED={"condition_label": "t", "record_id": "r", "motor_id": "m", "measured_at": "t"},
                CLASS_NAMES=P.CLASSES, split_motors=split, train=train, val=val, test=test, X_train=Xtr, y_train=train.y.to_numpy(),
                X_val_s=scaler.transform(val[feats].to_numpy(float)), y_val=val.y.to_numpy(), scaler=scaler, MIN_CLASS_RECORDS=10)


def test_clean_pipeline_passes(tmp_path):
    out = N.run_notebook_leakage_checks(namespace(tmp_path), n_perm=3)
    assert set(out[out.check.isin(["N1", "N2", "N3", "N4", "N5"])].result) == {"PASS"}


def test_scaler_fitted_on_all_data_fails(tmp_path):
    with pytest.raises(AssertionError, match="N1"):
        N.run_notebook_leakage_checks(namespace(tmp_path, fit_on="all"), n_perm=2)


def test_label_in_features_fails(tmp_path):
    with pytest.raises(AssertionError) as e:
        N.run_notebook_leakage_checks(namespace(tmp_path, leak_label_feature=True), n_perm=3)
    assert "N4" in str(e.value) or "N5" in str(e.value)


## Step 2. Point to your data
If the file is not found, a Colab upload box appears. Choose your **original** CSV. Nothing is changed or re-saved.

In [ ]:
import os, glob, hashlib
DATA = os.environ.get("PM_DATA_PATH", "/content/motor_predictive_maintenance_master.csv")
if not os.path.exists(DATA):
    try:
        from google.colab import files
        up = files.upload()                       # choose motor_predictive_maintenance_master.csv
        DATA = "/content/" + next(iter(up))
    except ImportError:
        raise FileNotFoundError("Set PM_DATA_PATH to the original CSV.")
CLEAN  = os.environ.get("PM_CLEAN_COPY") or (glob.glob("/content/outputs/motor_clean_working_copy.csv") or [None])[0]   # optional
BUNDLE = os.environ.get("PM_BUNDLE_PATH") or (sorted(glob.glob("/content/outputs/pm_condition_ann_v*/"))[-1:] or [None])[0]  # optional
print("data  :", DATA, "\nclean :", CLEAN, "\nbundle:", BUNDLE)
print("SHA-256 of the original (write this down; it must never change):", hashlib.sha256(open(DATA,'rb').read()).hexdigest())

## Step 3. Run the audit
Takes about 20 to 60 seconds. Read the list from the top: **FAIL** = must be fixed or explained, **WARN** = a question for you or the data owner, **info** = context, **pass** = checked and fine.

In [ ]:
cmd = f'python pm_data_audit.py --data "{DATA}" --out /content/audit_outputs'
if CLEAN:  cmd += f' --clean-copy "{CLEAN}"'
if BUNDLE: cmd += f' --bundle "{BUNDLE}"'
print(cmd, "\n")
!{cmd}

## Step 4. See the full detail
The table lists every FAIL and WARN with the evidence behind it.

In [ ]:
import json, pandas as pd
from IPython.display import display
rep = json.load(open("/content/audit_outputs/audit_report.json"))
print("VERDICT:", rep["verdict"])
pd.set_option("display.max_colwidth", 400, "display.width", 250)
tbl = pd.DataFrame(rep["findings"])[["id","status","title","detail"]]
display(tbl[tbl.status.isin(["FAIL","WARN"])])
for name, t in rep["tables"].items():
    print("\n##", name); display(pd.DataFrame(t) if isinstance(t, list) else pd.DataFrame(t).T)

## Step 5. Run the automated tests on your real files
`PASSED` means the check ran on your data and held. `SKIPPED` means it did **not** run (a file was not provided). `XFAIL` on the failure-target test is expected: it documents that your file has no failure-event column.

In [ ]:
env = f'PM_DATA_PATH="{DATA}"' + (f' PM_CLEAN_COPY="{CLEAN}"' if CLEAN else '') + (f' PM_BUNDLE_PATH="{BUNDLE}"' if BUNDLE else '')
!{env} python -m pytest tests -v -rsx -p no:cacheprovider 2>&1 | tail -80

## Step 6. Add the leakage checks to your training notebook (recommended)
Open `notebook_leakage_checks.py` (left Files panel), copy all of it into a **new cell in your training notebook, right after Section 12 (scaler fitted) and before Section 13 (training)**, and run it. It stops the notebook if it finds a leak (scaler fitted on all data, a label/ID/time column among the inputs, a split that mixes motors) and prints warnings about the limits of your claim.

## Step 7. Save the reports (separate from your original data)

In [ ]:
!cd /content && zip -qr audit_outputs.zip audit_outputs
try:
    from google.colab import files; files.download("/content/audit_outputs.zip")
except ImportError:
    print("Reports are in /content/audit_outputs")